In [0]:
%pip install yfinance

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import yfinance as yf
import pandas as pd
import pyspark

In [0]:
tickers = ["JPM", "BAC", "GS", "C", "WFC"]
df = yf.download(tickers, start="2022-01-01", end="2024-01-01", group_by="ticker")

df = df.stack(level=0).reset_index()
df.columns = ["Date", "Ticker", "Close", "High", "Low", "Open", "Volume"]

df["Date"] = pd.to_datetime(df["Date"])
df["Volume"] = df["Volume"].astype("int64")

print(f"Shape: {df.shape}")
print(f"Tickers: {df["Ticker"].unique()}")
df.head()

[*********************100%***********************]  5 of 5 completed

Shape: (2505, 7)
Tickers: ['BAC' 'C' 'GS' 'JPM' 'WFC']



/home/spark-67997760-cc98-4bbd-99ee-0a/.ipykernel/3922/command-7584009730755054-915001270:4: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(level=0).reset_index()


,Date,Ticker,Close,High,Low,Open,Volume
0,2022-01-03,BAC,40.500685,41.758193,40.401881,41.479744,58587900
1,2022-01-03,C,52.464488,54.548318,52.404461,54.110970,30508500
2,2022-01-03,GS,349.655941,358.914172,348.163833,355.345703,3334300
3,2022-01-03,JPM,143.280341,145.772016,142.966635,144.929504,13120900
4,2022-01-03,WFC,43.986161,45.676209,43.977169,45.604290,38978100


In [0]:
spark_df = spark.createDataFrame(df)

spark_df.write.format("delta").mode("overwrite").saveAsTable("bronze.bronze_stocks_prices")
print("Saved")

spark.sql("SELECT count(*) as total_rows FROM bronze_stocks_prices").show()
spark.sql("SELECT Date, Ticker, Close, High, Low, Open, Volume FROM bronze_stocks_prices").show()

Saved
+----------+
|total_rows|
+----------+
|      2505|
+----------+

+-------------------+------+------------------+------------------+------------------+------------------+--------+
|               Date|Ticker|             Close|              High|               Low|              Open|  Volume|
+-------------------+------+------------------+------------------+------------------+------------------+--------+
|2022-01-03 00:00:00|   BAC|40.500689064935756| 41.75819685597981|40.401884440812324|  41.4797477722168|58587900|
|2022-01-03 00:00:00|     C| 52.46448832387503|54.548318366404835|52.404460567198655| 54.11096954345703|30508500|
|2022-01-03 00:00:00|    GS|349.65585105278427| 358.9140798383467|348.16374310104754| 355.3456115722656| 3334300|
|2022-01-03 00:00:00|   JPM| 143.2803105731878|145.77198508194508|142.96660493686375|144.92947387695312|13120900|
|2022-01-03 00:00:00|   WFC| 43.98616461312506| 45.67621235355437|43.977173082203905| 45.60429382324219|38978100|
|2022-01-04 00:0